In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
# Check missing values in each column
df.isnull().sum()

# isnull() checks every cell and says whether it is missing.
# sum() counts how many missing values each column has.
# note: machine learning models usually cannot work with missing values.

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [8]:
# Fill missing values
# We need to separate categorical and numerical columns, so we can populate them differently.    
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [11]:
 list(df.select_dtypes(include=["str"]).columns)

['lead_source', 'industry', 'employment_status', 'location']

In [12]:

numerical = list(df.select_dtypes(include=["int64", "float64"]).columns)

numerical.remove("converted") #this is the target variable, not a feature, so remove it.

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1
### What is the most frequent observation (mode) for the column industry?

- NA
- technology
- healthcare
- retail

In [15]:
df["industry"]

0          technology
1          technology
2              retail
3       manufacturing
4       manufacturing
            ...      
4995        education
4996       technology
4997          finance
4998       technology
4999       technology
Name: industry, Length: 5000, dtype: str

In [16]:
df["industry"].mode()[0] #mode return the most common industry. It is a list so take the first.

'technology'

## Question 2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

Only consider the pairs above when answering this question.

In [17]:
corr_matrix = df[numerical].corr()
corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


lead score and interaction count have highest, at 0.91

## Split the Data
### Split into train, validation, and test

In [19]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

We split the data into three parts:
- Training set: used to train the model.
- Validation set: used to check how good the model is.
- Test set: saved for final evaluation.
  
20% of the data goes into the test set.

Then we split the remaining 80% again:

test_size=0.25

25% of 80% is 20%, so we end up with:

60% training, 20% validation, 20% test

random_state=42
so that everyone gets the same split. Without it, the random split could be different every time.

In [21]:
# Separate the target from the features
y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

converted what we want to predict.
The features are all the other columns.
So we store the target separately:
y_train
y_val
y_test

Then we delete converted from the feature dataframes.
Again, this prevents the model from cheating by seeing the answer during training.

## Question 3
Calculate the mutual information score between converted and other categorical variables in the dataset.

Use the training set only.

Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?

- industry
- location
- lead_source
- employment_status

In [25]:
for col in categorical:
    score = mutual_info_score(y_train, df_train[col])
    print(col, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


The question asks which categorical variable is most useful for predicting converted.
    
For categorical variables, correlation is not a good choice because correlation works mainly with numbers.

So we use:
mutual_info_score()

Mutual information measures how much knowing one variable tells us about another variable.

Here we ask:
If we know the value of this categorical column, how much does it help us understand converted?

A higher score means the feature is more informative.

Answer:

lead_source

## Question 4
Now let's train a logistic regression.

Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.

Fit the model on the training dataset.

To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
- model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [23]:
features = categorical + numerical

train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)

round(accuracy, 2)

0.65

We combine all features:

features = categorical + numerical

The model needs both categorical and numerical information.

But logistic regression cannot directly understand text like:
- technology
- retail
- organic_search

So we convert each row into a dictionary:

to_dict(orient="records")

Then we use:

DictVectorizer()

This converts categorical values into one-hot encoded columns.

For example, a column like:

industry = technology

becomes something like:

- industry=technology: 1
- industry=retail: 0
- industry=healthcare: 0
- 
Then we train logistic regression:

model.fit(X_train, y_train)

After training, we predict on validation data:

y_pred = model.predict(X_val)

Finally, we compare predictions with the true answers:

accuracy_score(y_val, y_pred)

### Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
- 
Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.

In [24]:
original_accuracy = accuracy

for feature in features:
    smaller_feature_set = features.copy()
    smaller_feature_set.remove(feature)

    train_dicts = df_train[smaller_feature_set].to_dict(orient="records")
    val_dicts = df_val[smaller_feature_set].to_dict(orient="records")

    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy_without_feature = accuracy_score(y_val, y_pred)

    difference = original_accuracy - accuracy_without_feature

    print(feature, accuracy_without_feature, difference)

lead_source 0.642 0.0030000000000000027
industry 0.645 0.0
employment_status 0.645 0.0
location 0.645 0.0
annual_income 0.724 -0.07899999999999996
number_of_courses_viewed 0.643 0.0020000000000000018
interaction_count 0.601 0.04400000000000004
lead_score 0.644 0.0010000000000000009


Feature elimination means:

Remove one feature at a time and see how much the model gets worse.

If removing a feature barely changes accuracy, then that feature is probably not very useful.

If removing a feature makes accuracy much worse, then that feature was important.

We compare:

original accuracy

against:

accuracy without one feature

The difference tells us how much that feature mattered.

Among the options, the smallest difference is for:

number_of_courses_viewed


### Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001

> **Note**: If there are multiple options, select the smallest `C`.

In [26]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    train_dicts = df_train[features].to_dict(orient="records")
    val_dicts = df_val[features].to_dict(orient="records")

    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    print(C, round(accuracy, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


C controls regularization strength in logistic regression.

Regularization is a way to stop the model from relying too strongly on the training data.

Important idea:
- Smaller C means stronger regularization.
- Larger C means weaker regularization.
- 
The homework asks us to try these values:
[0.000001, 0.00001, 0.0001, 0.001]
Then we choose the one with the best validation accuracy.

Answer:

0.001